# Debate / Society of Mind

> Notebook generated from [`examples/patterns/02_debate.py`](../../examples/patterns/02_debate.py).

| Field | Value |
|------|-------|
| **Dataset** | BoolQ + AI ethics dilemmas (embedded) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** For each question: debate rounds per agent (proponent/opponent), moderator consensus and Jaccard agreement score (0–1).

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Debate / Society of Mind — AI ethics and business decisions
==================================================================
Pattern: SPEC-PAT-002 / prismal.agents.patterns.debate

Dataset: BoolQ + custom ethics questions
  • BoolQ: 15,942 yes/no questions extracted from Wikipedia.
  • Reference: https://huggingface.co/datasets/google/boolq
  • Why: The Debate pattern generates multiple perspectives on ambiguous
    or controversial claims, exactly the type of question where BoolQ
    and ethics questions add value.

Pattern description:
  N agents with different roles (proponent, opponent, neutral, analyst)
  debate a question over M rounds. In each round, agents see prior positions
  and refine their own. The moderator synthesizes the final consensus.
  Agreement is measured with Jaccard similarity over the token sets.

Usage:
    uv run python examples/patterns/02_debate.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio

from prismal.agents.patterns.debate import DebateResult, debate_round

## Dataset: selected debate topics

In [ ]:
# We combine BoolQ questions (controversial) with AI ethics dilemmas.
DEBATE_TOPICS = [
    {
        "query": (
            "Should artificial intelligence be used to make hiring "
            "decisions without human oversight?"
        ),
        "category": "ai_ethics",
        "n_agents": 3,
        "roles": ["proponent", "opponent", "neutral"],
        "n_rounds": 2,
    },
    {
        "query": (
            "Do large language models (LLMs) pose an existential risk "
            "to humanity within the next 20 years?"
        ),
        "category": "existential_risk",
        "n_agents": 4,
        "roles": ["techno_optimist", "existential_pessimist", "pragmatist", "ethicist"],
        "n_rounds": 3,
    },
    {
        "query": (
            "Is it ethical for tech companies to monetize their users' "
            "personal data to train AI models?"
        ),
        "category": "data_privacy",
        "n_agents": 3,
        "roles": ["company_advocate", "user_advocate", "regulator"],
        "n_rounds": 2,
    },
    {
        "query": (
            "Should the source code of frontier AI models be open source "
            "to guarantee public safety?"
        ),
        "category": "open_source_AI",
        "n_agents": 3,
        "roles": ["safety_researcher", "company_executive", "academic"],
        "n_rounds": 2,
    },
]


def print_separator(char: str = "─", width: int = 70) -> None:
    print(char * width)


def print_result(topic: dict, result: DebateResult) -> None:
    """Print the result of a debate in a structured way."""
    print_separator("═")
    print(f"  Category : {topic['category']}")
    print(f"  Question : {topic['query'][:80]}...")
    print(f"  Agents   : {topic['n_agents']} | Rounds: {topic['n_rounds']}")
    print("  Strategy : moderator")
    print_separator()

    # Show positions per round
    for ronda in range(1, topic["n_rounds"] + 1):
        ronda_positions = [p for p in result.positions if p.round == ronda]
        if ronda_positions:
            print(f"\n  [Round {ronda}]")
            for pos in ronda_positions:
                print(f"    [{pos.role}] {pos.content[:120]}...")

    # Final consensus
    print("\n  [FINAL CONSENSUS]")
    print(f"  {result.consensus}")

    # Metrics
    print(f"\n  Agreement (Jaccard): {result.agreement_score:.3f}", end="")
    if result.agreement_score > 0.6:
        print("  ✓ High consensus")
    elif result.agreement_score > 0.3:
        print("  ~ Moderate consensus")
    else:
        print("  ✗ Low consensus / high divergence")

    if result.dissenting_views:
        print(f"\n  Dissenting views ({len(result.dissenting_views)}):")
        for dv in result.dissenting_views[:2]:
            print(f"    • {dv[:100]}...")

    print(f"\n  Rounds completed: {result.rounds_completed}")
    print_separator("─")


async def run_debate(topic: dict) -> DebateResult:
    """Run a debate for a given topic."""
    return await debate_round(
        query=topic["query"],
        state={},  # opaque state — reserved for future extensions
        n_agents=topic["n_agents"],
        n_rounds=topic["n_rounds"],
        roles=topic["roles"],
        synthesis_strategy="moderator",
    )


async def main() -> None:
    print("\n" + "═" * 70)
    print("  Debate / Society of Mind — Dataset: AI Ethics + BoolQ-style")
    print("═" * 70)

    results = []

    for i, topic in enumerate(DEBATE_TOPICS, 1):
        print(f"\n>>> Debate {i}/{len(DEBATE_TOPICS)}: {topic['category']}")
        result = await run_debate(topic)
        results.append((topic, result))
        print_result(topic, result)

    # Comparative summary
    print("\n" + "═" * 70)
    print("  COMPARATIVE SUMMARY")
    print("═" * 70)
    print(f"  {'Category':<30} {'Agents':>7} {'Rounds':>6} {'Agreement':>9}")
    print("  " + "─" * 60)
    for topic, res in results:
        print(
            f"  {topic['category']:<30} "
            f"{topic['n_agents']:>7} "
            f"{res.rounds_completed:>6} "
            f"{res.agreement_score:>8.3f}"
        )

    # Topic with highest agreement
    best = max(results, key=lambda x: x[1].agreement_score)
    worst = min(results, key=lambda x: x[1].agreement_score)
    print(f"\n  Highest agreement : {best[0]['category']} ({best[1].agreement_score:.3f})")
    print(f"  Largest dispute   : {worst[0]['category']} ({worst[1].agreement_score:.3f})")


if __name__ == "__main__":
    asyncio.run(main())


---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()